In [ ]:
# --- Imports ---
import sympy as sp
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

# --- 1. Define symbolic coordinates and metric ---
theta, phi = sp.symbols('theta phi')
metric_sphere = sp.Matrix([
    [1, 0],
    [0, sp.sin(theta)**2]
])

coords = [theta, phi]
n = len(coords)

# --- 2. Compute Christoffel symbols ---
Gamma = [[[0]*n for _ in range(n)] for _ in range(n)]

for k in range(n):
    for i in range(n):
        for j in range(n):
            Gamma[k][i][j] = sum(
                sp.Rational(1,2) * sp.simplify(
                    sp.diff(metric_sphere[k,l], coords[i])
                    + sp.diff(metric_sphere[k,l], coords[j])
                    - sp.diff(metric_sphere[i,j], coords[l])
                ) * sp.Matrix(metric_sphere).inv()[l,k]
                for l in range(n)
            )

# --- 3. Lambdify Christoffel symbols ---
Gamma_num = [[[sp.lambdify((theta, phi), Gamma[k][i][j], 'numpy') for j in range(n)] for i in range(n)] for k in range(n)]

# --- 4. Geodesic ODE ---
def geodesic_equations(lam, y):
    x = y[:n]      # theta, phi
    v = y[n:]      # dtheta/dlambda, dphi/dlambda
    dxdl = v
    dvdl = np.zeros_like(v)
    
    for mu in range(n):
        acc = 0.0
        for nu in range(n):
            for rho in range(n):
                acc += -Gamma_num[mu][nu][rho](x[0], x[1]) * v[nu] * v[rho]
        dvdl[mu] = acc
    
    return np.concatenate([dxdl, dvdl])

# --- 5. Initial conditions ---
theta0 = np.pi / 2      # start at equator
phi0 = 0.0
dtheta0 = 0.0           # moving along equator
dphi0 = 1.0

y0 = np.array([theta0, phi0, dtheta0, dphi0])

# --- 6. Integrate ---
lam_span = (0, 10)
t_eval = np.linspace(*lam_span, 500)
sol = solve_ivp(geodesic_equations, lam_span, y0, t_eval=t_eval, rtol=1e-9)

# --- 7. Convert to 3D coordinates for plotting ---
theta_vals = sol.y[0]
phi_vals = sol.y[1]

x_vals = np.sin(theta_vals) * np.cos(phi_vals)
y_vals = np.sin(theta_vals) * np.sin(phi_vals)
z_vals = np.cos(theta_vals)

# --- 8. Plot ---
fig = plt.figure(figsize=(6,6))
ax = fig.add_subplot(111, projection='3d')

# Sphere
u = np.linspace(0, np.pi, 50)
v = np.linspace(0, 2*np.pi, 50)
X = np.outer(np.sin(u), np.cos(v))
Y = np.outer(np.sin(u), np.sin(v))
Z = np.outer(np.cos(u), np.ones_like(v))
ax.plot_surface(X, Y, Z, color='c', alpha=0.2)

# Geodesic
ax.plot(x_vals, y_vals, z_vals, 'r', lw=2)

ax.set_box_aspect([1,1,1])
plt.show()